### Choose where this notebook runs

This is the **IQM edition**: IQM **Garnet** (20 qubits) and **Emerald** (54 qubits) have no classical feedforward,
so the dynamic circuits of this notebook run on the local **Aer** simulator; the final section shows the
`TranspilerError` that `transpile_to_IQM` gives for an `if_test` circuit and runs the deferred-measurement
form of a conditional gate (a CNOT before the final measurements) on the IQM backend chosen below, in the native gate set **PRX + CZ**. Pick the backend
by editing the single `TARGET` line below (see `README.md` and `iqm_backend.py` in this folder):

| `TARGET` | runs on | token? | QPU credits? |
|---|---|---|---|
| `"fake_apollo"` | local 20-qubit noisy simulator | no | no |
| `"garnet:mock"` / `"emerald:mock"` | IQM Resonance cloud mock (pipeline check: returns RANDOM bitstrings, no physics) | yes | no |
| `"garnet"` / `"emerald"` | **real QPU** (20q / 54q) | yes | yes |


# Classical Feedforward and Control Flow

**Course**: Quantum Computing Laboratory
**Version**: 2.2 (Qiskit 2.x Compatible)
**Last Updated**: October 2026

---


In [ ]:
from iqm_backend import get_backend, run_counts   # helper module in this folder (see iqm_backend.py)
# run_counts(circuit, TARGET, shots=...) = transpile to the IQM native gates (PRX + CZ) + run + get_counts()
# get_backend(TARGET) returns the raw Qiskit backend object if you want to call backend.run yourself
TARGET = "fake_apollo"   # change this one line to choose the backend (see the table above)


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `from iqm_backend import get_backend, run_counts` | the helper module of this folder: `run_counts(circuit, TARGET, shots=...)` = `transpile_to_IQM` (rewrite the circuit with the native `r` = PRX rotations and `cz` gates, routed on the chip's square lattice) → `backend.run` → counts as a plain dict; `run_counts(..., return_transpiled=True)` also returns the transpiled circuit |
| `TARGET = "fake_apollo"` | a local noisy simulator with the 20-qubit Garnet topology: no token, no credits, realistic gate and readout noise; `"garnet:mock"` checks the cloud pipeline but returns RANDOM bitstrings; `"garnet"` / `"emerald"` are the real QPUs (token + credits; IQM Resonance declines circuits above 10 000 native instructions) |
| `get_backend(TARGET)` | the raw Qiskit backend object (name, qubit count, coupling map) if you want to call `backend.run` yourself |

</details>


## 📚 Overview

This guide demonstrates the functionality available in the Qiskit SDK for performing classical feedforward and control flow. These features are sometimes referred to collectively as "dynamic circuits." Classical feedforward refers to the ability to measure qubits in the middle of a circuit and perform additional quantum operations that depend on the measurement outcome. Qiskit supports four control flow constructs for classical feedforward, each implemented as a method on [`QuantumCircuit`](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.circuit.QuantumCircuit). The constructs and their corresponding methods are:

*   If statement - [`QuantumCircuit.if_test`](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.circuit.QuantumCircuit#if_test)
*   Switch statement - [`QuantumCircuit.switch`](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.circuit.QuantumCircuit#switch)
*   For loop - [`QuantumCircuit.for_loop`](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.circuit.QuantumCircuit#for_loop)
*   While loop - [`QuantumCircuit.while_loop`](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.circuit.QuantumCircuit#while_loop)

Each of these methods returns a [context manager](https://docs.python.org/3/reference/datamodel.html#with-statement-context-managers) and is typically used in a `with` statement. In the rest of this guide, we will explain each of these constructs and how to use them.

> ⚠️ **Caution.** There are some limitations of classical feedforward and control flow operations on quantum hardware that might impact your program. For more information, see [Hardware considerations and limitations for classical feedforward and control flow](https://quantum.cloud.ibm.com/docs/guides/dynamic-circuits-considerations).

In this notebook every circuit is not only drawn but also **executed** on the local Aer simulator, which supports
dynamic circuits, and the counts are checked against a hand calculation. A final section adds classical
**expressions** (`qiskit.circuit.classical.expr`) as conditions.

### Topics Covered:
- mid-circuit measurement and `if_test` (with `else`), on one bit or on a whole register
- `switch` with several cases and a default
- `for_loop` and `while_loop`
- conditions built from classical expressions (`expr.bit_xor`, …)


## 🎯 Learning Objectives

1. ✅ Measure in the middle of a circuit and condition later gates on the result
2. ✅ Write `if`/`else`, `switch`, `for` and `while` blocks with the `with` statement
3. ✅ Predict the output distribution of a dynamic circuit by summing over the measurement branches
4. ✅ Know the reset-by-measurement and repeat-until-success patterns


## 🔧 Prerequisites

**Required Knowledge:**
- Measurement, classical registers and bit ordering (`05_Measurement.ipynb`)
- Python `with` blocks (context managers)

**Required Software:**
- Python 3.9+
- Qiskit 2.x, Qiskit Aer (≥ 0.13, for control flow and classical expressions)

**Recommended Previous Notebooks:**
- `05_Measurement.ipynb`


In [ ]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister   # circuits built from named registers
from qiskit import transpile                             # rewrite a circuit into the gates a backend supports
from qiskit.circuit.classical import expr                # classical expressions for conditions (last section)
from qiskit_aer import AerSimulator                      # local simulator: supports mid-circuit measurement and control flow
import numpy as np                                       # pi (In-depth analysis)
from IPython.display import display                      # show a figure (e.g. a circuit drawing) from the middle of a cell
def draw(qc, fold=-1, **kw): return qc.draw("mpl", style="clifford", fold=fold, **kw)   # every circuit picture: matplotlib drawer; fold=-1 = one long row, fold=N wraps after N columns

# IQM edition: IQM backends (fake_apollo, garnet, emerald) have NO classical feedforward, so the dynamic circuits
# of this notebook stay on Aer. Only the deferred-measurement circuits of the IQM section run on TARGET (run_counts).
aer = AerSimulator()                                     # one simulator object, reused by every run
def run_circuit(circuit, shots=1024):
    """Run a (dynamic) circuit on the Aer simulator and return the counts {bitstring: number of shots}."""
    return aer.run(transpile(circuit, aer), shots=shots).result().get_counts()   # transpile: rewrite gates Aer lacks (e.g. ch); .result() waits; .get_counts() -> dict

print("✓ imports done — dynamic circuits run on", aer.name, "| IQM section runs on TARGET =", TARGET)


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `QuantumRegister`, `ClassicalRegister` | registers as objects: the examples unpack them into individual `Qubit` / `Clbit` handles (`q0`, `c0`) that the control-flow methods take as conditions |
| `from qiskit.circuit.classical import expr` | builders of classical expressions (`bit_xor`, `logic_and`, `equal`, `less`, …) that `if_test` / `while_loop` accept as conditions |
| `AerSimulator()` + `run_circuit` — **not** `TARGET` | in this IQM edition the dynamic circuits run on the local Aer simulator: the IQM chips execute a fixed list of gates and measure at the end, with no classical feedforward, and `transpile_to_IQM` rejects `if_test`, `switch` and loops. The section *Dynamic circuits on IQM hardware* shows the error and the deferred-measurement circuit that does run on `TARGET` |
| `draw(qc)` | our helper around `qc.draw("mpl", style="clifford")`; control-flow blocks are drawn as boxes with their condition |

</details>


> 🖥️ **IQM note.** The dynamic circuits of the next sections (mid-circuit measurement followed by `if_test`, `switch`,
> `for_loop`, `while_loop`) are executed on the local **Aer** simulator through `run_circuit`, not on `TARGET`:
> IQM Garnet / Emerald and their fake backends have no classical feedforward. What *can* run on IQM — the
> deferred-measurement form of a conditional gate — is shown in the section **Dynamic circuits on IQM hardware** at the end.


## If statement


The if statement is used to conditionally perform operations based on the value of a classical bit or register.

In the example below, we apply a Hadamard gate to a qubit and measure it. If the result is 1, then we apply an X gate on the qubit, which has the effect of flipping it back to the 0 state. We then measure the qubit again. The resulting measurement outcome should be 0 with 100% probability.


In [ ]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister

qubits = QuantumRegister(1)                    # a register of 1 qubit (automatic name 'q0', 'q1', ... — a new one per register)
clbits = ClassicalRegister(1)                  # a register of 1 classical bit
print(qubits)                                  # QuantumRegister(size, 'name')
circuit = QuantumCircuit(qubits, clbits)       # a circuit built from registers instead of integer counts
(q0,) = qubits                                 # tuple unpacking: a register is a sequence of Qubit objects; (x,) = a 1-tuple
(c0,) = clbits                                 # same for the single Clbit

circuit.h(q0)                                  # |0> -> (|0> + |1>)/sqrt2
circuit.measure(q0, c0)                        # MID-circuit measurement: c0 = 0 or 1 with probability 1/2, the qubit collapses
with circuit.if_test((c0, 1)):                 # if_test((bit, value)): the indented block runs only in shots where c0 == 1
    circuit.x(q0)                              # flips the collapsed |1> back to |0>
circuit.measure(q0, c0)                        # second measurement, into the same bit
display(draw(circuit)); print("↑ H, a mid-circuit measurement, an if-block (X only when c0 = 1), and a final measurement")

counts = run_circuit(circuit)
print("counts:", counts)
# example output counts: {'0': 1024}


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `(q0,) = qubits` | unpacks the 1-element register into its only `Qubit`; writing `q0 = qubits[0]` is equivalent |
| `circuit.measure(q0, c0)` before other gates | a **mid-circuit measurement**: unlike a final one, the circuit continues on the collapsed state, and its result is available to later instructions in the same shot |
| `with circuit.if_test((c0, 1)):` | everything indented under the `with` becomes the body of an `IfElseOp` instruction executed only when the condition `c0 == 1` holds *in that shot*; the drawing shows it as a box labelled with the condition |
| `counts` → `{'0': 1024}` | every shot ends in 0: whichever result the first measurement gave, the qubit is \|0⟩ afterwards — a reset built from measurement and feedforward |
| **IQM edition** | this cell runs on the local Aer simulator (`run_circuit`), not on `TARGET`: IQM backends have no classical feedforward and reject `if_test` / `switch` / loops — see *Dynamic circuits on IQM hardware* |

</details>


<details open><summary><b>📐 Working it out</b></summary>

**The reset-by-measurement trick.** After H the qubit is $\tfrac{1}{\sqrt2}(|0\rangle+|1\rangle)$; the first measurement
gives $c_0=0$ with probability $\tfrac12$ (state $|0\rangle$) or $c_0=1$ with probability $\tfrac12$ (state $|1\rangle$).
Follow each branch:

| branch | probability | state after measuring | `if_test` | state before final measure | final $c_0$ |
|---|---|---|---|---|---|
| $c_0=0$ | $1/2$ | $\lvert 0\rangle$ | skipped | $\lvert 0\rangle$ | 0 |
| $c_0=1$ | $1/2$ | $\lvert 1\rangle$ | $X$ applied | $\lvert 0\rangle$ | 0 |

So $P(\text{final }0)=\tfrac12\cdot1+\tfrac12\cdot1=1$: all 1024 shots read `'0'`, for *any* input state. This is how
hardware implements an "active reset" (Qiskit's `circuit.reset(q0)` has the same effect); a unitary gate alone could
never do it, because mapping both $|0\rangle$ and $|1\rangle$ to $|0\rangle$ is not reversible.

</details>


The `with` statement can be given an assignment target which is itself a context manager that can be stored and subsequently used to create an else block, which is executed whenever the contents of the if block are *not* executed.

In the example below, we initialize registers with two qubits and two classical bits. We apply a Hadamard gate to the first qubit and measure it. If the result is 1, then we apply a Hadamard gate on the second qubit; otherwise, we apply an X gate on the second qubit. Finally, we measure the second qubit as well.

In [ ]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1) = qubits                              # unpack the two Qubit objects
(c0, c1) = clbits                              # and the two Clbit objects

circuit.h(q0)                                  # q0 -> |+>
circuit.measure(q0, c0)                        # c0 = 0 or 1, probability 1/2 each
with circuit.if_test((c0, 1)) as else_:        # `as else_` keeps a handle to build the else branch
    circuit.h(q1)                              # c0 == 1: put q1 in |+> -> c1 random
with else_:                                    # the else block: runs exactly when the if block does not
    circuit.x(q1)                              # c0 == 0: q1 -> |1> -> c1 = 1
circuit.measure(q1, c1)

display(draw(circuit)); print("↑ one IfElseOp box with two bodies: H on q1 when c0 = 1, X on q1 otherwise")
counts = run_circuit(circuit)
print("counts:", counts)                       # keys 'c1 c0' (clbit 0 on the right)
# example output counts: {'01': 260, '11': 272, '10': 492}


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `with circuit.if_test(...) as else_:` | `if_test` returns a context manager; binding it with `as` gives a second context manager for the `else` branch, so the two branches form a single `if_else` instruction |
| `with else_:` | must come immediately after the if block; it runs in the shots where `c0 == 0` |
| `counts` keys `'c1c0'` | `'10'` = c0 = 0 (else branch, X made c1 = 1), `'01'` and `'11'` = c0 = 1 (if branch, H made c1 random). Never `'00'`: c0 = 0 always comes with c1 = 1 |
| **IQM edition** | this cell runs on the local Aer simulator (`run_circuit`), not on `TARGET`: IQM backends have no classical feedforward and reject `if_test` / `switch` / loops — see *Dynamic circuits on IQM hardware* |

</details>


<details open><summary><b>📐 Working it out</b></summary>

Sum over the two branches of the first measurement:

* $c_0=0$ (prob. $\tfrac12$): the else branch applies $X$, $q_1=|1\rangle$, so $c_1=1$ with certainty → key `'10'` with $P=\tfrac12$.
* $c_0=1$ (prob. $\tfrac12$): the if branch applies $H$, $q_1=|+\rangle$, so $c_1=0$ or $1$ with $\tfrac12$ each → keys `'01'` and `'11'` with $P=\tfrac12\cdot\tfrac12=\tfrac14$ each.

Expected counts out of 1024: `'10'` ≈ 512, `'01'` ≈ 256, `'11'` ≈ 256 — the example output `{'01': 260, '11': 272, '10': 492}` fits within shot noise ($\sigma\approx\sqrt{1024\cdot\tfrac14\cdot\tfrac34}\approx14$).

</details>


In addition to conditioning on a single classical bit, it's also possible to condition on the value of a classical register composed of multiple bits.

In the example below, we apply Hadamard gates to two qubits and measure them. If the result is `01`, that is, the first qubit is 1 and the second qubit is 0, then we apply an X gate to a third qubit. Finally, we measure the third qubit. Note that for clarity, we chose to specify the state of the third classical bit, which is 0, in the if condition. In the circuit drawing, the condition is indicated by the circles on the classical bits being conditioned on. A black circle indicates conditioning on 1, while a white circle indicates conditioning on 0.

In [ ]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])                            # two random bits
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.if_test((clbits, 0b001)):         # condition on the WHOLE register: (c2 c1 c0) == 001, i.e. c0 = 1, c1 = 0, c2 = 0
    circuit.x(q2)
circuit.measure(q2, c2)

display(draw(circuit, cregbundle=False)); print("↑ the if-box is conditioned on the 3-bit register (drawn bit by bit): filled circle = must be 1, open circle = must be 0")   # cregbundle=False: one wire per classical bit
counts = run_circuit(circuit)
print("counts:", counts)

# example output counts: {'101': 269, '011': 260, '000': 252, '010': 243}


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `if_test((clbits, 0b001))` | a `(register, integer)` condition compares the register's value with the integer; `0b001` is Python's binary literal (= 1), bit 0 on the right like the keys |
| c2 inside the condition | c2 is still 0 when the test runs (it has not been measured yet), so the condition `001` is the same as "c0 = 1 and c1 = 0"; the author spells out c2 = 0 for clarity |
| `counts` → 4 keys | `'000'`, `'010'`, `'011'` (no X) and `'101'` (X fired, c2 = 1), about 1/4 each; `'001'` never appears because whenever c1c0 = 01 the X turns c2 into 1 |
| **IQM edition** | this cell runs on the local Aer simulator (`run_circuit`), not on `TARGET`: IQM backends have no classical feedforward and reject `if_test` / `switch` / loops — see *Dynamic circuits on IQM hardware* |

</details>


<details open><summary><b>📐 Working it out</b></summary>

**Probability bookkeeping of a conditional X.** The two Hadamards make $c_1c_0$ uniformly random: each of $00, 01, 10, 11$
has probability $\tfrac14$. The X on $q_2$ fires only in the branch where the register reads $c_2c_1c_0=001$:

| $c_1c_0$ | $P$ | condition $001$? | $q_2$ | final key $c_2c_1c_0$ |
|---|---|---|---|---|
| 00 | 1/4 | no | $\lvert 0\rangle$ | `000` |
| 01 | 1/4 | **yes** → X | $\lvert 1\rangle$ | `101` |
| 10 | 1/4 | no | $\lvert 0\rangle$ | `010` |
| 11 | 1/4 | no | $\lvert 0\rangle$ | `011` |

So $P(c_2=1)=P(c_1c_0=01)=\tfrac14$: about 256 of the 1024 shots read `'101'`, as in the example output (269).

</details>


## Switch statement

The switch statement is used to select actions based on the value of a classical bit or register. It is similar to an if statement, but allows one to specify more cases for the branching logic. In the example below, we apply a Hadamard gate to a qubit and measure it. If the result is 0, we apply an X gate on the qubit, and if the result is 1, we apply a Z gate. The resulting measurement outcome should be 1 with 100% probability.

In [ ]:
qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.switch(c0) as case:               # switch on the value of c0; `case` builds the individual branches
    with case(0):                              # c0 == 0: the qubit collapsed to |0>
        circuit.x(q0)                          #   -> |1>
    with case(1):                              # c0 == 1: the qubit is |1>
        circuit.z(q0)                          #   Z|1> = -|1>: a phase only, still reads 1
circuit.measure(q0, c0)

display(draw(circuit)); print("↑ a SwitchCaseOp box on c0 with two cases: X for 0, Z for 1")
counts = run_circuit(circuit)
print("counts:", counts)
# example output counts: {'1': 1024}


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `with circuit.switch(c0) as case:` | opens a switch on a bit or register; `case` is a function returning one context manager per branch |
| `with case(0):`, `with case(1):` | the body of each branch; exactly one runs per shot |
| `circuit.z(q0)` on \|1⟩ | Z only multiplies \|1⟩ by −1, a global phase that no measurement can see: both branches end in \|1⟩, so every shot reads `'1'` |
| **IQM edition** | this cell runs on the local Aer simulator (`run_circuit`), not on `TARGET`: IQM backends have no classical feedforward and reject `if_test` / `switch` / loops — see *Dynamic circuits on IQM hardware* |

</details>


Because the example above used a single classical bit, there were only two possible cases, so we could have achieved the same result using an if-else statement. The switch case is mainly useful when branching on the value of a classical register composed of multiple bits, as demonstrated in the following example. Here, we also show how to construct a default case, which is executed if none of the preceding cases are. Note that in a switch statement, only one of the blocks are ever executed. There is no fallthrough.

In the example below, we apply Hadamard gates to two qubits and measure them. If the result is either 00 or 11, we apply a Z gate to the third qubit. If the result is 01, we apply a Y gate. If none of the preceding cases matched, we apply an X gate. Finally, measure the third qubit.

In [ ]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.switch(clbits) as case:           # switch on the integer value of the whole 3-bit register
    with case(0b000, 0b011):                   # several values may share one branch
        circuit.z(q2)                          # Z|0> = |0>: c2 stays 0
    with case(0b001):
        circuit.y(q2)                          # Y|0> = i|1>: c2 = 1
    with case(case.DEFAULT):                   # every other value (here only 0b010)
        circuit.x(q2)                          # X|0> = |1>: c2 = 1
circuit.measure(q2, c2)

display(draw(circuit)); print("↑ a switch on the 3-bit register with three branches: {000, 011} -> Z, {001} -> Y, default -> X")
counts = run_circuit(circuit)
print("counts:", counts)
# example output counts: {'101': 267, '110': 249, '011': 265, '000': 243}


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `case(0b000, 0b011)` | one branch for several values; a value may appear in only one case |
| `case(case.DEFAULT)` | the fallback branch, run when no other case matches; there is no fall-through between cases |
| Z, Y, X on \|0⟩ | Z leaves \|0⟩ (c2 = 0), Y gives i\|1⟩ and X gives \|1⟩ (c2 = 1). So c1c0 = 00 → `'000'`, 01 → `'101'`, 10 → `'110'`, 11 → `'011'`, each with probability 1/4 |
| **IQM edition** | this cell runs on the local Aer simulator (`run_circuit`), not on `TARGET`: IQM backends have no classical feedforward and reject `if_test` / `switch` / loops — see *Dynamic circuits on IQM hardware* |

</details>


## For loop

A for loop is used to iterate over a sequence of classical values and perform some operations during each iteration.

In the example below, we use a for loop to apply 5 X gates to a qubit and then measure it. Because we perform an odd number of X gates, the overall effect is to flip the qubit from the 0 state to the 1 state.

In [ ]:
qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

with circuit.for_loop(range(5)) as _:          # repeat the body for each value in range(5); `_` = loop variable we do not use
    circuit.x(q0)
circuit.measure(q0, c0)

display(draw(circuit)); print("↑ a ForLoopOp box: its body (one X) is executed five times")
counts = run_circuit(circuit)
print("counts:", counts)                       # X^5 = X: always '1'
# example output counts: {'1': 1024}


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `with circuit.for_loop(range(5)) as _:` | a `ForLoopOp`: the body is stored once and repeated for each value of the iterable; `as i` would give a `Parameter` usable inside the body (e.g. as a rotation angle) |
| `X^5 = X` | since X² = I, an odd number of X gates is one X: \|0⟩ → \|1⟩, every shot reads `'1'` |
| why not a Python loop? | `for _ in range(5): circuit.x(q0)` would write 5 separate X gates into the circuit; `for_loop` keeps the loop in the circuit itself, which matters for long loops and for hardware that executes loops natively |
| **IQM edition** | this cell runs on the local Aer simulator (`run_circuit`), not on `TARGET`: IQM backends have no classical feedforward and reject `if_test` / `switch` / loops — see *Dynamic circuits on IQM hardware* |

</details>


## While loop

A while loop is used to repeat instructions while some condition is satisfied.

In the example below, we apply Hadamard gates to two qubits and measure them. Then, we create a while loop that repeats this procedure while the measurement outcome is 11. As a result, the final measurement should never be 11, with the remaining possibilities appearing with approximately equal frequency.

In [ ]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)

q0, q1 = qubits
c0, c1 = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.while_loop((clbits, 0b11)):       # repeat the body WHILE the register reads 11
    circuit.h([q0, q1])                        # H on a collapsed basis state gives a fresh uniform superposition
    circuit.measure(q0, c0)
    circuit.measure(q1, c1)

display(draw(circuit)); print("↑ a WhileLoopOp box conditioned on (c1 c0) == 11: H + measurements are repeated until the result differs from 11")
counts = run_circuit(circuit, shots=3000)
print("counts:", counts)
print({k: round(v / 3000, 3) for k, v in sorted(counts.items())}, "  <- each close to 1/3")   # dict comprehension: relative frequencies
# example output counts: {'01': 334, '10': 368, '00': 322}


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `with circuit.while_loop((clbits, 0b11)):` | a `WhileLoopOp`: the condition is checked before every iteration; the body re-measures the register, so the loop ends at the first result different from 11 (*repeat until success*) |
| `circuit.h([q0, q1])` inside the loop | after a measurement each qubit is \|0⟩ or \|1⟩; H turns either into an equal superposition (up to a sign), so every round is a fresh uniform draw of two bits |
| `shots=3000` | 3000 instead of 1024 to make the 1/3 frequencies easier to see; `'11'` never occurs |
| **IQM edition** | this cell runs on the local Aer simulator (`run_circuit`), not on `TARGET`: IQM backends have no classical feedforward and reject `if_test` / `switch` / loops — see *Dynamic circuits on IQM hardware* |

</details>


<details open><summary><b>📐 Working it out</b></summary>

Each round gives $11$ with probability $q=\tfrac14$ (repeat) and each of $00, 01, 10$ with probability $\tfrac14$ (stop).
The outcome $00$ is obtained after exactly $k$ failed rounds with probability $q^k\cdot\tfrac14$, so
$$P(00)=\sum_{k=0}^{\infty}\Big(\tfrac14\Big)^k\tfrac14=\frac{1/4}{1-1/4}=\frac13,$$
and the same for $01$ and $10$: the loop turns the uniform distribution over four outcomes into the uniform
distribution over three. The number of rounds is geometric with success probability $\tfrac34$, mean
$1/(3/4)=4/3$ rounds. Expected frequencies: $0.333$ each, i.e. about 1000 of the 3000 shots per key.

</details>


## Classical expressions (`expr`)

The conditions above compare a single bit or a whole register with a constant. The module
`qiskit.circuit.classical.expr` builds richer run-time conditions from bits and registers: logic operations
(`logic_and`, `logic_or`, `logic_not`), bitwise operations (`bit_and`, `bit_or`, `bit_xor`, `bit_not`) and comparisons
(`equal`, `not_equal`, `less`, `greater`, …). Any expression of Boolean type can be passed to `if_test` or
`while_loop` in place of a `(bit, value)` tuple.

In the example below we measure two random bits and apply an X to a third qubit when their **parity** is odd,
$c_0 \oplus c_1 = 1$, so that the three bits always have even parity.


In [ ]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
parity = expr.bit_xor(c0, c1)                  # an Expr node: c0 XOR c1, evaluated at run time in each shot
print("condition:", parity, "| type:", parity.type)   # the expression tree and its classical type (Bool)
with circuit.if_test(parity):                  # an Expr of type Bool replaces the (bit, value) tuple
    circuit.x(q2)                              # odd parity -> flip q2
circuit.measure(q2, c2)

display(draw(circuit)); print("↑ an if-box whose condition is the expression c0 XOR c1")
counts = run_circuit(circuit)
print("counts:", counts)
print("every key has an even number of 1s:", all(k.count("1") % 2 == 0 for k in counts))   # all(...) over a generator


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `expr.bit_xor(c0, c1)` | builds an expression *object* (it is not evaluated in Python): the control hardware or simulator evaluates it in each shot after the measurements |
| `parity.type` | every expression has a classical type; `if_test` needs `Bool`. Comparisons such as `expr.equal(clbits, 5)` or `expr.less(clbits, 2)` also give `Bool` |
| `counts` → `'000'`, `'011'`, `'101'`, `'110'` | c2 = c0 ⊕ c1 in every shot, about 1/4 each: the circuit copies the parity of two random bits into a third one |
| `all(k.count('1') % 2 == 0 for k in counts)` | a one-line check of the even-parity property on every observed key |
| **IQM edition** | this cell runs on the local Aer simulator (`run_circuit`), not on `TARGET`: IQM backends have no classical feedforward and reject `if_test` / `switch` / loops — see *Dynamic circuits on IQM hardware* |

</details>


## 🖥️ Dynamic circuits on IQM hardware

IQM Garnet and Emerald execute a fixed list of PRX and CZ gates and read all qubits at the **end** of the circuit:
there is no mid-circuit feedforward, and `transpile_to_IQM` rejects every control-flow instruction. What can be run
is the **deferred-measurement** form of a conditional gate. The circuit

> H on $q_0$ → measure $q_0\to c_0$ → *if* $c_0=1$: X on $q_1$ → measure $q_1\to c_1$

has exactly the same output statistics as

> H on $q_0$ → **CNOT** $q_0\to q_1$ → measure $q_0\to c_0$ and $q_1\to c_1$,

in which the classically controlled X has become a quantum-controlled X placed *before* the measurement. The cell
below (1) tries to transpile the `if_test` circuit for `TARGET` and prints the error, (2) runs the conditional circuit
on Aer and the deferred circuit on `TARGET` with `run_counts`.


In [ ]:
from iqm.qiskit_iqm import transpile_to_IQM            # the IQM transpiler that run_counts uses internally

# (a) the dynamic circuit: measure, then a classically controlled X
cond = QuantumCircuit(2, 2)
cond.h(0)
cond.measure(0, 0)                             # mid-circuit measurement of q0
with cond.if_test((cond.clbits[0], 1)):        # X on q1 only in the shots where c0 == 1
    cond.x(1)
cond.measure(1, 1)

try:
    transpile_to_IQM(cond, backend=get_backend(TARGET))   # IQM has no feedforward: this raises
    print("transpiled without error (unexpected)")
except Exception as err:                       # TranspilerError: the control-flow construct 'if_else' is not supported
    print(f"if_test circuit on {TARGET} -> {type(err).__name__}: {err}")

# (b) the deferred-measurement form: the classical control becomes a CNOT before the final measurements
deferred = QuantumCircuit(2, 2)
deferred.h(0)
deferred.cx(0, 1)                              # quantum-controlled X replaces "measure, then if c0 == 1: X"
deferred.measure([0, 1], [0, 1])               # all measurements at the end: no feedforward needed
display(draw(cond)); print("↑ (a) the dynamic circuit: H, mid-circuit measurement, an if-box with X on q1, final measurement")
display(draw(deferred)); print("↑ (b) its deferred-measurement form: H, CNOT, then both measurements at the end")

shots = 2048
counts_cond = run_circuit(cond, shots=shots)   # Aer: the dynamic circuit itself
counts_def, deferred_iqm = run_counts(deferred, TARGET, shots=shots, return_transpiled=True)   # IQM: the deferred circuit
print(f"deferred circuit transpiled for IQM: {dict(deferred_iqm.count_ops())}, depth {deferred_iqm.depth()}")   # 'r' = PRX, 'cz'
for key in ("00", "01", "10", "11"):
    print(f"  {key}: if_test on Aer {counts_cond.get(key, 0) / shots:.3f}   CNOT on {TARGET} {counts_def.get(key, 0) / shots:.3f}")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `transpile_to_IQM(cond, backend=...)` in `try/except` | the IQM transpiler refuses the circuit before anything runs: `TranspilerError: The control-flow construct 'if_else' is not supported by the backend.` The same happens for `switch_case`, `for_loop` and `while_loop`, and on the real `garnet` / `emerald` |
| `deferred.cx(0, 1)` | the deferred-measurement replacement of *measure q0, then if c0 = 1: X on q1*: a CNOT controlled by the qubit itself, followed by the measurements |
| `run_counts(deferred, TARGET, ..., return_transpiled=True)` | the deferred circuit through the full IQM pipeline: H and CNOT become `r` (PRX) rotations and one `cz`, routed on two neighbouring qubits of the square lattice |
| the two columns | ideally 0.5 for `00` and `11` and 0 for `01`, `10` in both: the dynamic circuit on Aer is exact, the deferred one on `fake_apollo` puts a few per cent on `01` / `10` because of gate and readout noise |
| what is lost | the *statistics* are identical, but the deferred circuit keeps q0 coherent until the end: it cannot reuse q0 after the measurement (no qubit recycling) and it cannot express a while loop, whose number of rounds is unbounded |

</details>


<details open><summary><b>📐 Working it out</b></summary>

**Why the CNOT is equivalent.** Write the state after H as $\tfrac{1}{\sqrt2}(|0\rangle+|1\rangle)_{q_0}|0\rangle_{q_1}$.

*Dynamic circuit.* Measuring $q_0$ gives $c_0=0$ with probability $\tfrac12$ (then nothing happens, $q_1=|0\rangle$,
key `00`) or $c_0=1$ with probability $\tfrac12$ (then X, $q_1=|1\rangle$, key `11`): $P(00)=P(11)=\tfrac12$.

*Deferred circuit.* The CNOT acts before any measurement:
$$\text{CNOT}\,\tfrac{1}{\sqrt2}(|00\rangle+|10\rangle)_{q_0q_1}=\tfrac{1}{\sqrt2}(|00\rangle+|11\rangle),$$
and the Born rule gives again $P(00)=P(11)=\tfrac12$.

In general a gate controlled by the classical result of measuring qubit $a$ equals the same gate quantum-controlled
by $a$ followed by the measurement of $a$ (*principle of deferred measurement*, Nielsen & Chuang §4.4): the
controlled gate is block-diagonal in the computational basis of $a$, $C_U = |0\rangle\langle0|\otimes I + |1\rangle\langle1|\otimes U$,
so it commutes with the projectors $|0\rangle\langle0|$, $|1\rangle\langle1|$ of the Z measurement on $a$. Both columns of
the cell should therefore show $\approx 0.5,\,0,\,0,\,0.5$ — exactly on Aer, up to the noise of `fake_apollo` on IQM.

</details>


## 🔬 In-depth analysis

1. **Active reset of any state.** Prepare $R_y(\theta)|0\rangle$ for $\theta = \pi/3,\ 2\pi/3,\ \pi$, measure it into one classical bit, apply the `if_test` X of the first example, then measure again into a *second* bit. Check that the first bit is 1 with probability $\sin^2(\theta/2)$ while the second bit is always 0.
2. **A `for_loop` and the Born rule.** Apply $R_y(\pi/4)$ to a qubit $n$ times with `for_loop(range(n))`, for $n = 1, 2, 3, 4$, and compare the measured P(1) with $\sin^2(n\pi/8)$.
3. **Deferring an if/else.** The if/else example (H on $q_1$ if $c_0=1$, X on $q_1$ otherwise) can also be written without feedforward: a controlled-H (`ch(0, 1)`) for the if branch and an X controlled on $q_0 = 0$ (`x(0); cx(0, 1); x(0)`) for the else branch, followed by the two measurements. Build it and compare its distribution with the `if_test` version (total variation distance).

Write your code in the empty cell below; a reference solution follows it. Try first, then compare.


In [ ]:
# 🔬 In-depth analysis — your code here



In [ ]:
# Reference solution
shots = 4000

# (1) active reset: first bit random with P(1) = sin^2(θ/2), second bit always 0
for theta in (np.pi / 3, 2 * np.pi / 3, np.pi):
    qc = QuantumCircuit(1, 2)                  # one qubit, two classical bits: c0 = first reading, c1 = after the reset
    qc.ry(theta, 0)
    qc.measure(0, 0)
    with qc.if_test((qc.clbits[0], 1)):        # qc.clbits = the list of Clbit objects of the circuit
        qc.x(0)
    qc.measure(0, 1)
    c = run_circuit(qc, shots=shots)           # keys 'c1 c0' -> k[1] = c0, k[0] = c1
    p_first = sum(n for k, n in c.items() if k[1] == '1') / shots
    p_second = sum(n for k, n in c.items() if k[0] == '1') / shots
    print(f"(1) θ = {theta:.3f}: P(first = 1) = {p_first:.3f} (sin²(θ/2) = {np.sin(theta / 2)**2:.3f}), P(after reset = 1) = {p_second:.3f}")

# (2) for_loop of Ry(π/4): n rotations add up to Ry(nπ/4)
for n in range(1, 5):
    qc = QuantumCircuit(1, 1)
    with qc.for_loop(range(n)):
        qc.ry(np.pi / 4, 0)
    qc.measure(0, 0)
    p1 = run_circuit(qc, shots=shots).get('1', 0) / shots
    print(f"(2) n = {n}: P(1) = {p1:.3f}   sin²(nπ/8) = {np.sin(n * np.pi / 8)**2:.3f}")

# (3) the if/else example and its deferred-measurement version
dyn = QuantumCircuit(2, 2)
dyn.h(0); dyn.measure(0, 0)
with dyn.if_test((dyn.clbits[0], 1)) as else_:
    dyn.h(1)
with else_:
    dyn.x(1)
dyn.measure(1, 1)

dfr = QuantumCircuit(2, 2)
dfr.h(0)
dfr.ch(0, 1)                                   # if q0 = 1: H on q1
dfr.x(0); dfr.cx(0, 1); dfr.x(0)               # if q0 = 0: X on q1 (X-sandwich = control on |0>)
dfr.measure([0, 1], [0, 1])

p_dyn = {k: v / shots for k, v in run_circuit(dyn, shots=shots).items()}
p_dfr = {k: v / shots for k, v in run_circuit(dfr, shots=shots).items()}
keys = sorted(set(p_dyn) | set(p_dfr))         # union of the observed keys
tvd = 0.5 * sum(abs(p_dyn.get(k, 0) - p_dfr.get(k, 0)) for k in keys)   # total variation distance
print("(3) if/else :", {k: round(p_dyn.get(k, 0), 3) for k in keys})
print("    deferred:", {k: round(p_dfr.get(k, 0), 3) for k in keys}, f"  TVD = {tvd:.3f} (shot noise only)")
display(draw(dfr)); print("↑ task 3: the if/else example without feedforward — controlled-H, then an X controlled on q0 = 0, then both measurements")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `QuantumCircuit(1, 2)` + `measure(0, 0)` … `measure(0, 1)` | two classical bits for one qubit: the first reading is kept in c0, the reading after the reset goes to c1, so the key `'c1c0'` shows both |
| `with qc.for_loop(range(n)):` | n rotations by π/4 about y compose to one rotation by nπ/4, so P(1) = sin²(nπ/8): 0.146, 0.5, 0.854, 1 for n = 1…4 |
| `x(0); cx(0, 1); x(0)` | a CNOT that fires when q0 is \|0⟩: the X gates swap the roles of \|0⟩ and \|1⟩ of the control and are undone afterwards |
| `tvd = 0.5 * Σ\|p − q\|` | the total variation distance between two distributions (0 = identical, 1 = disjoint); a value of order 0.01 is just the shot noise of two 4000-shot samples |
| **IQM edition** | this cell runs on the local Aer simulator (`run_circuit`), not on `TARGET`: IQM backends have no classical feedforward and reject `if_test` / `switch` / loops — see *Dynamic circuits on IQM hardware* |

</details>


## Next steps

> 💡 **Recommendations**
>
> *   See an example of dynamic circuits in the [Repeat until success](https://quantum.cloud.ibm.com/docs/tutorials/repeat-until-success) tutorial.
> *   Review the [circuit library API](https://quantum.cloud.ibm.com/docs/api/qiskit/circuit_library) reference.

---

**Course**: Quantum Computing Laboratory
**License**: MIT
**Version**: 2.2.0
**Last Updated**: October 2026
